# data and temporal split

The validation period starts at the same cutoff as the EDA. Training rows must finish their 90-day target window before validation starts.

In [1]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if not (root / "data").exists():
    root = root.parent

train = pd.read_csv(root / "data/processed/train.csv")
dates = pd.to_datetime(train["purchase_date"])
validation_start = dates.sort_values().iloc[int(len(train) * 0.8)]
fit_rows = dates + pd.Timedelta(days=90) < validation_start
validation_rows = dates >= validation_start

print("Validation starts:", validation_start.date())
print("Training rows:", fit_rows.sum(), "| Validation rows:", validation_rows.sum())

Validation starts: 2023-06-17
Training rows: 5024 | Validation rows: 1298


# baseline features

These features use current purchase totals and past purchase activity. Negative recency is treated as missing. Median imputation is fitted inside each model pipeline.

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

features = ["Total a Pagar", "days_since_previous_purchase", "prior_purchase_count",
            "prior_mean_purchase_value", "purchase_count_90d", "spend_90d"]
X = train[features].copy()
recency = X["days_since_previous_purchase"]
X["days_since_previous_purchase"] = recency.mask(recency < 0)

repeat = train["repeat_purchase_90d"]
spend = train["spend_next_90d"]
has_spend = repeat.eq(1) & spend.notna()

# classification baseline

Compare a class-prior dummy with a small Random Forest. The 0.5 threshold is fixed for this baseline.

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, roc_auc_score

classifier = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=42),
)
classifiers = {"Dummy": DummyClassifier(), "Random Forest": classifier}
class_scores = {}
for name, model in classifiers.items():
    model.fit(X.loc[fit_rows], repeat.loc[fit_rows])
    probability = model.predict_proba(X.loc[validation_rows])[:, 1]
    decision = probability >= 0.5
    actual = repeat.loc[validation_rows]
    class_scores[name] = {
        "ROC AUC": roc_auc_score(actual, probability),
        "Average precision": average_precision_score(actual, probability),
        "Accuracy": accuracy_score(actual, decision),
        "F1": f1_score(actual, decision),
    }

display(pd.DataFrame(class_scores).T.round(3))

               ROC AUC  Average precision  Accuracy     F1
Dummy            0.500              0.603     0.603  0.753
Random Forest    0.755              0.831     0.690  0.762


# regression baseline

Train and evaluate on actual repeat purchases with observed spending. This measures the regressor separately from classifier errors.

In [4]:
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

regressor = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    RandomForestRegressor(n_estimators=100, min_samples_leaf=5, random_state=42),
)
regressors = {"Train median": DummyRegressor(strategy="median"), "Random Forest": regressor}
fit_spend = fit_rows & has_spend
validation_spend = validation_rows & has_spend
reg_scores = {}
for name, model in regressors.items():
    model.fit(X.loc[fit_spend], spend.loc[fit_spend])
    estimate = model.predict(X.loc[validation_spend])
    actual = spend.loc[validation_spend]
    reg_scores[name] = {
        "RMSE": root_mean_squared_error(actual, estimate),
        "MAE": mean_absolute_error(actual, estimate),
        "R2": r2_score(actual, estimate),
    }

print("Observed spending targets in validation:", validation_spend.sum())
display(pd.DataFrame(reg_scores).T.round(3))

Observed spending targets in validation: 694
                   RMSE       MAE     R2
Train median   3494.879  1764.867 -0.146
Random Forest  2691.297  1521.054  0.321


# combined predictions

Produce an example spending estimate only for purchases predicted to repeat. The regression metrics above use the actual repeat labels.

In [5]:
validation_X = X.loc[validation_rows]
probability = classifier.predict_proba(validation_X)[:, 1]
predictions = train.loc[validation_rows, ["ID"]].copy()
predictions["repeat_probability"] = probability
predictions["repeat_purchase_90d"] = (probability >= 0.5).astype(int)
predictions["spend_next_90d"] = float("nan")

predicted_return = predictions["repeat_purchase_90d"].eq(1)
predictions.loc[predicted_return, "spend_next_90d"] = regressor.predict(validation_X.loc[predicted_return])
print("Predicted repeat purchases:", predicted_return.sum(), "of", len(predictions))
display(predictions.head().round(2))

Predicted repeat purchases: 909 of 1298
          ID  repeat_probability  repeat_purchase_90d  spend_next_90d
0   10000000                0.46                    0             NaN
5   10000005                0.78                    1         2105.42
11  10000014                0.26                    0             NaN
18  10000024                0.78                    1         1584.43
19  10000026                0.19                    0             NaN
